In [ ]:
 !pip install "numpy<2.0"

In [ ]:
!pip install --upgrade scikit-surprise

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import polars as pl
import pandas as pd
from surprise import SVD, Dataset, Reader, accuracy
from collections import defaultdict
import numpy as np

# --- 1. SETUP & DATA LOADING ---
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
split_dir = f"{drive_path}/processed_data/final_foundation/splits"

# Load all three splits using Polars
train_pl = pl.read_parquet(f"{split_dir}/train.parquet")
val_pl = pl.read_parquet(f"{split_dir}/val.parquet")
test_pl = pl.read_parquet(f"{split_dir}/test.parquet")


In [10]:
# --- 2. PREPARE DATA FOR SURPRISE ---
# Memory Management: Sample for training to fit Colab T4 RAM limits
train_sample = train_pl.sample(n=2_000_000, seed=42).to_pandas()
val_pd = val_pl.to_pandas()
test_pd = test_pl.to_pandas()

reader = Reader(rating_scale=(1, 10))
# Load training data
data = Dataset.load_from_df(train_sample[['user_id', 'item_id', 'Rating']], reader)
trainset = data.build_full_trainset()

# Prepare validation and test tuples for Surprise .test() method
val_tuples = list(val_pd[['user_id', 'item_id', 'Rating']].itertuples(index=False, name=None))
test_tuples = list(test_pd[['user_id', 'item_id', 'Rating']].itertuples(index=False, name=None))

In [11]:
# --- 3. HYPERPARAMETER TUNING (Validation Set) ---
# Tuning 'n_factors' as an example of using the validation set
n_factors_options = [20, 50, 100]
best_n_factors = None
best_val_rmse = float('inf')

print("--- Tuning 'n_factors' on Validation Set ---")
for factors in n_factors_options:
    algo = SVD(n_factors=factors, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
    algo.fit(trainset)

    # Predict on Validation set
    val_predictions = algo.test(val_tuples)
    val_rmse = accuracy.rmse(val_predictions, verbose=False)

    print(f"n_factors={factors:3} | Val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_n_factors = factors

print(f"\nOptimal n_factors selected: {best_n_factors}")

--- Tuning 'n_factors' on Validation Set ---
n_factors= 20 | Val RMSE: 1.2458
n_factors= 50 | Val RMSE: 1.2566
n_factors=100 | Val RMSE: 1.2641

Optimal n_factors selected: 20


In [12]:
# --- 4. EVALUATION FUNCTIONS ---
def get_metrics_at_k(predictions, k=10, threshold=7.0):
    """Return RMSE, Precision, Recall, and NDCG at k."""
    user_est_true = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = dict()
    recalls = dict()
    ndcgs = []

    for uid, user_ratings in user_est_true.items():
        # Sort by estimate for P, R, and DCG
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # P and R logic
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold))
                              for (est, true_r) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / k if k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

        # NDCG logic
        top_k_est = [true_r for (_, true_r) in user_ratings[:k]]
        user_ratings.sort(key=lambda x: x[1], reverse=True) # Sort by true for IDCG
        top_k_ideal = [true_r for (_, true_r) in user_ratings[:k]]

        def dcg(ratings):
            return sum([r / np.log2(i + 2) for i, r in enumerate(ratings)])

        idcg = dcg(top_k_ideal)
        if idcg > 0:
            ndcgs.append(dcg(top_k_est) / idcg)

    return {
        "RMSE": accuracy.rmse(predictions, verbose=False),
        "Precision@10": np.mean(list(precisions.values())),
        "Recall@10": np.mean(list(recalls.values())),
        "NDCG@10": np.mean(ndcgs)
    }

In [ ]:
# --- 5. FINAL EVALUATION (Test Set) ---
# Train final model with the best hyperparameter found
print(f"\nTraining final SVD model with n_factors={best_n_factors}...")
final_algo = SVD(n_factors=best_n_factors, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
final_algo.fit(trainset)

# Execute final metrics on Test Set
test_predictions = final_algo.test(test_tuples)
final_test_metrics = get_metrics_at_k(test_predictions, k=10, threshold=7.0)

print(f"\n--- Final SVD Baseline Results (Test Set, n_factors={best_n_factors}) ---")
for metric, value in final_test_metrics.items():
    print(f"{metric:12}: {value:.4f}")


Training final SVD model with n_factors=20...

--- Final SVD Baseline Results (Test Set, n_factors=20) ---
RMSE        : 1.2960
Precision@10: 0.2297
Recall@10   : 0.7563
NDCG@10     : 0.9834
